# FlowGuard - Phase 1: MLP Baseline

In [ ]:
import os, sys
os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
sys.path.insert(0, '.')

from src.utils.config import load_config, get_device
from src.utils.reproducibility import setup_reproducibility
from src.models.mlp_baseline import MLPBaseline
from src.data.dataset import create_dataloader
from src.data.features import get_input_dim
from src.training.supervised import SupervisedTrainer
from src.evaluation.protocols import evaluate_protocol_a, evaluate_protocol_b

config = load_config("configs/phase1_baseline.yaml")
setup_reproducibility(config)
device = get_device(config)
print(f"Device: {device}")

## Train on each dataset (Protocol A)

In [ ]:
results_a = {}
for ds_info in config['data']['datasets']:
    name = ds_info['name']
    train_path = f"data/splits/protocol_a/{name}_train.parquet"
    val_path = f"data/splits/protocol_a/{name}_val.parquet"
    
    if not os.path.exists(train_path):
        print(f"Skipping {name}")
        continue
    
    print(f"\n{'='*50}\nTraining MLP on: {name}\n{'='*50}")
    
    train_loader = create_dataloader(train_path, batch_size=1024, balanced=True)
    val_loader = create_dataloader(val_path, batch_size=1024, shuffle=False)
    
    model = MLPBaseline(input_dim=get_input_dim(), hidden_dims=[256, 128, 64], num_classes=2, dropout=0.3)
    trainer = SupervisedTrainer(model, train_loader, val_loader, config, checkpoint_dir=f"checkpoints/phase1/{name}/")
    history = trainer.train()
    
    results_a[name] = history

## Protocol A Evaluation

In [ ]:
import torch
for ds_info in config['data']['datasets']:
    name = ds_info['name']
    ckpt_path = f"checkpoints/phase1/{name}/best_model.pt"
    if not os.path.exists(ckpt_path):
        continue
    
    model = MLPBaseline(input_dim=get_input_dim(), hidden_dims=[256, 128, 64], num_classes=2)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(device)
    
    print(f"\n--- {name} ---")
    evaluate_protocol_a(model, config, device)